# 🌊 The Dam That Chooses Between Flood Protection and Water Security
## Civil Engineering + Deep Learning with an MLP

Predict reservoir level six hours after a forecast storm, then select a pre-release that creates flood storage without ignoring downstream river condition or stored-water security.

> Educational decision-support simulation only. This notebook does not authorize or recommend operation of a real dam.

👉 **Open the interactive companion:** [https://dam-flood-water-security.streamlit.app](https://dam-flood-water-security.streamlit.app/?stage=start)

## Six inputs and one model output

| Input | Unit |
|---|---:|
| Current reservoir level | % |
| Forecast rainfall | mm |
| Upstream inflow | m³/s |
| Downstream river level | encoded Low–Very high |
| Reservoir capacity remaining | % |
| Current release rate | m³/s |

**MLP output:** predicted reservoir level after six hours. A separate rule-based layer selects and constrains the pre-release.

## Interactive learning journey

- [One Reservoir, Two Duties](https://dam-flood-water-security.streamlit.app/?stage=dilemma) — Multi-Objective Decision
- [Six Conditions Before the Storm](https://dam-flood-water-security.streamlit.app/?stage=inputs) — Input Features
- [Recorded Storm Events](https://dam-flood-water-security.streamlit.app/?stage=data) — Training Dataset
- [Putting Measurements on One Scale](https://dam-flood-water-security.streamlit.app/?stage=prepare) — Normalization
- [Forecasting the Six-Hour Level](https://dam-flood-water-security.streamlit.app/?stage=mlp) — Feedforward Neural Network
- [Selecting a Pre-Release](https://dam-flood-water-security.streamlit.app/?stage=decision) — Constrained Decision Layer
- [Wait Versus Act Early](https://dam-flood-water-security.streamlit.app/?stage=compare) — Scenario Simulation
- [The Dam Safety Audit](https://dam-flood-water-security.streamlit.app/?stage=audit) — Model and Policy Evaluation

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Input,Dense,Dropout
from tensorflow.keras.callbacks import EarlyStopping
SEED=42;np.random.seed(SEED);tf.random.set_seed(SEED)
FEATURES=["level_pct","rain_mm","inflow_m3s","downstream_code","free_storage_pct","current_release_m3s"]
print("Six summarized inputs → MLP → six-hour reservoir level")

---
# 1. One Reservoir, Two Duties
### Phase 1 of 6 · The Operating Dilemma

## Part 1 · At the dam
Before a forecast storm, the dam stores water for supply, irrigation, and power while also needing empty space to absorb inflow.

## Part 2 · Engineering challenge
Release too little and an emergency spill may be required later. Release too much and valuable storage is lost if the forecast is wrong.

## Part 3 · Where AI comes in
Prediction supplies an early view of the reservoir level expected after six hours; a separate engineering rule turns that forecast into a cautious release.

**Civil Engineering:** One Reservoir, Two Duties → **AI:** Multi-Objective Decision → `flood safety vs conserved storage`

> 🎬 **See this illustrated and interactive:** [https://dam-flood-water-security.streamlit.app/?stage=dilemma](https://dam-flood-water-security.streamlit.app/?stage=dilemma)

## Part 4 · Technical explanation

A pre-release creates storage before storm inflow arrives. Its benefit is avoiding a late emergency spill; its cost is water that may not be recovered if the forecast overstates the event. The model predicts one physical quantity. The operating balance remains a transparent decision problem.

## Part 5 · What you just built

**In the notebook:** Define the operating objective and the no-pre-release baseline.

**Takeaway:** The goal is not maximum release or maximum storage; it is a defensible balance.

[Project overview](https://dam-flood-water-security.streamlit.app/?stage=start) &nbsp;|&nbsp; [Next: Six Conditions Before the Storm](https://dam-flood-water-security.streamlit.app/?stage=inputs) ▶

---
# 2. Six Conditions Before the Storm
### Phase 2 of 6 · Reading the Catchment

## Part 1 · At the dam
Operators know how full the reservoir is, expected rainfall, current inflow, downstream river state, remaining storage, and current release.

## Part 2 · Engineering challenge
Each variable changes the consequence of the others: a high reservoir with low rainfall is different from the same level before an extreme storm.

## Part 3 · Where AI comes in
Use exactly six summarized inputs so the student model remains explainable and does not require a long time sequence.

**Civil Engineering:** Six Conditions Before the Storm → **AI:** Input Features → `level, rain, inflow, downstream, free storage, release`

> 🎬 **See this illustrated and interactive:** [https://dam-flood-water-security.streamlit.app/?stage=inputs](https://dam-flood-water-security.streamlit.app/?stage=inputs)

## Part 4 · Technical explanation

In [ ]:
example={"level_pct":88,"rain_mm":120,"inflow_m3s":420,"downstream_code":2,"free_storage_pct":12,"current_release_m3s":150}
pd.Series(example,name="Example pre-storm snapshot")

## Part 5 · What you just built

**In the notebook:** Generate synthetic event rows with the six inputs and a six-hour future-level target.

**Takeaway:** The input row is a compact engineering snapshot of the coming event.

◀ [Previous: One Reservoir, Two Duties](https://dam-flood-water-security.streamlit.app/?stage=dilemma) &nbsp;|&nbsp; [Project overview](https://dam-flood-water-security.streamlit.app/?stage=start) &nbsp;|&nbsp; [Next: Recorded Storm Events](https://dam-flood-water-security.streamlit.app/?stage=data) ▶

---
# 3. Recorded Storm Events
### Phase 2 of 6 · Reading the Catchment

## Part 1 · At the dam
Past events pair pre-storm conditions with observed inflow, release, evaporation, spill, and final reservoir level.

## Part 2 · Engineering challenge
A useful teaching dataset must obey storage balance; arbitrary random percentages can produce impossible reservoir behaviour.

## Part 3 · Where AI comes in
Generate examples from a simplified water-balance model, then reserve later events for testing.

**Civil Engineering:** Recorded Storm Events → **AI:** Training Dataset → `hydrologic mass-balance simulation`

> 🎬 **See this illustrated and interactive:** [https://dam-flood-water-security.streamlit.app/?stage=data](https://dam-flood-water-security.streamlit.app/?stage=data)

## Part 4 · Technical explanation

In [ ]:
def generate_events(n=5000,seed=42):
 rng=np.random.default_rng(seed);level=rng.uniform(55,96,n);rain=np.clip(rng.gamma(2.1,38,n),0,230);inflow=np.clip(55+2.4*rain+rng.normal(0,60,n),30,750);down=np.clip((.006*inflow+rng.normal(0,.65,n)).astype(int),0,3);free=100-level;release=np.clip(45+2.2*np.maximum(level-70,0)+rng.normal(0,35,n),0,450)
 # Simplified six-hour storage response; noise represents forecast/model uncertainty.
 storm=.042*rain+.0052*inflow*6-.0044*release*6+.55*down+rng.normal(0,.8,n)
 future=np.clip(level+storm,30,103)
 return pd.DataFrame(dict(level_pct=level,rain_mm=rain,inflow_m3s=inflow,downstream_code=down,free_storage_pct=free,current_release_m3s=release,future_level_pct=future))
data=generate_events();print(data.shape);data.head()

In [ ]:
# Conservation check in simplified percentage-storage units.
assert np.allclose(data.free_storage_pct,100-data.level_pct)
data.describe().T

## Part 5 · What you just built

**In the notebook:** Simulate storm volume and future storage from consistent units.

**Takeaway:** Synthetic data should still respect conservation of mass.

◀ [Previous: Six Conditions Before the Storm](https://dam-flood-water-security.streamlit.app/?stage=inputs) &nbsp;|&nbsp; [Project overview](https://dam-flood-water-security.streamlit.app/?stage=start) &nbsp;|&nbsp; [Next: Putting Measurements on One Scale](https://dam-flood-water-security.streamlit.app/?stage=prepare) ▶

---
# 4. Putting Measurements on One Scale
### Phase 3 of 6 · Learning Storm Response

## Part 1 · At the dam
Rainfall is measured in millimetres, flow in cubic metres per second, and levels in percentages.

## Part 2 · Engineering challenge
Raw numerical size can make a neural network treat units as importance, and fitting a scaler on test events leaks future information.

## Part 3 · Where AI comes in
Fit the scaler on training rows only and transform validation and test rows with those same parameters.

**Civil Engineering:** Putting Measurements on One Scale → **AI:** Normalization → `training-only StandardScaler`

> 🎬 **See this illustrated and interactive:** [https://dam-flood-water-security.streamlit.app/?stage=prepare](https://dam-flood-water-security.streamlit.app/?stage=prepare)

## Part 4 · Technical explanation

In [ ]:
# Time-ordered split: first 70% train, next 15% validate, last 15% test.
n=len(data);i1=int(.70*n);i2=int(.85*n)
train,val,test=data.iloc[:i1],data.iloc[i1:i2],data.iloc[i2:]
scaler=StandardScaler().fit(train[FEATURES])
X_train=scaler.transform(train[FEATURES]);X_val=scaler.transform(val[FEATURES]);X_test=scaler.transform(test[FEATURES])
y_train=train.future_level_pct.to_numpy();y_val=val.future_level_pct.to_numpy();y_test=test.future_level_pct.to_numpy()
print(X_train.shape,X_val.shape,X_test.shape)

## Part 5 · What you just built

**In the notebook:** Create time-ordered splits and scale six features without leakage.

**Takeaway:** The scaler is part of the prediction system and must travel with the model.

◀ [Previous: Recorded Storm Events](https://dam-flood-water-security.streamlit.app/?stage=data) &nbsp;|&nbsp; [Project overview](https://dam-flood-water-security.streamlit.app/?stage=start) &nbsp;|&nbsp; [Next: Forecasting the Six-Hour Level](https://dam-flood-water-security.streamlit.app/?stage=mlp) ▶

---
# 5. Forecasting the Six-Hour Level
### Phase 3 of 6 · Learning Storm Response

## Part 1 · At the dam
The decision uses one summarized snapshot rather than a long sensor sequence.

## Part 2 · Engineering challenge
The relation between rainfall, catchment inflow, storage, and release is nonlinear, especially near full reservoir conditions.

## Part 3 · Where AI comes in
An MLP combines the six inputs through dense layers and outputs one continuous future-level percentage.

**Civil Engineering:** Forecasting the Six-Hour Level → **AI:** Feedforward Neural Network → `6 -> Dense64 -> Dense32 -> Dense16 -> 1`

> 🎬 **See this illustrated and interactive:** [https://dam-flood-water-security.streamlit.app/?stage=mlp](https://dam-flood-water-security.streamlit.app/?stage=mlp)

## Part 4 · Technical explanation

In [ ]:
model=Sequential([Input(shape=(6,)),Dense(64,activation="relu"),Dropout(.10),Dense(32,activation="relu"),Dense(16,activation="relu"),Dense(1)])
model.compile(optimizer="adam",loss="mae",metrics=["mae"])
early=EarlyStopping(monitor="val_loss",patience=7,restore_best_weights=True)
history=model.fit(X_train,y_train,validation_data=(X_val,y_val),epochs=70,batch_size=64,callbacks=[early],verbose=0)
model.summary()
plt.figure(figsize=(9,4));plt.plot(history.history["loss"],label="train");plt.plot(history.history["val_loss"],label="validation");plt.xlabel("Epoch");plt.ylabel("MAE (%)");plt.grid(alpha=.2);plt.legend();plt.show()

## Part 5 · What you just built

**In the notebook:** Train a compact regression MLP with MAE loss and early stopping.

**Takeaway:** Use the simplest model architecture that matches the way the inputs are represented.

◀ [Previous: Putting Measurements on One Scale](https://dam-flood-water-security.streamlit.app/?stage=prepare) &nbsp;|&nbsp; [Project overview](https://dam-flood-water-security.streamlit.app/?stage=start) &nbsp;|&nbsp; [Next: Selecting a Pre-Release](https://dam-flood-water-security.streamlit.app/?stage=decision) ▶

---
# 6. Selecting a Pre-Release
### Phase 4 of 6 · Choosing a Release

## Part 1 · At the dam
A high forecast level suggests creating space, but the river below the dam may already be high and unable to receive a large release safely.

## Part 2 · Engineering challenge
A model forecast alone does not decide what is operationally permissible downstream.

## Part 3 · Where AI comes in
Map predicted level to a proposed release, then cap it according to downstream river condition. Keep this rule visible and editable.

**Civil Engineering:** Selecting a Pre-Release → **AI:** Constrained Decision Layer → `forecast band + downstream cap`

> 🎬 **See this illustrated and interactive:** [https://dam-flood-water-security.streamlit.app/?stage=decision](https://dam-flood-water-security.streamlit.app/?stage=decision)

## Part 4 · Technical explanation

In [ ]:
def release_decision(predicted_level,downstream_code):
    proposed=0 if predicted_level<85 else 120 if predicted_level<90 else 250 if predicted_level<=95 else 400
    downstream_caps={0:450,1:320,2:250,3:180} # project assumptions for this simulation
    return min(proposed,downstream_caps[int(downstream_code)]),proposed

sample=pd.DataFrame([example]);pred=float(model.predict(scaler.transform(sample[FEATURES]),verbose=0)[0,0])
recommended,unconstrained=release_decision(pred,example["downstream_code"])
print(f"Predicted level after 6 hours: {pred:.1f}%")
print("Unconstrained release band:",unconstrained,"m³/s")
print("Downstream-constrained release:",recommended,"m³/s")

## Part 5 · What you just built

**In the notebook:** Apply project-assumption bands and downstream release caps.

**Takeaway:** Prediction estimates what may happen; engineering constraints govern what may be done.

◀ [Previous: Forecasting the Six-Hour Level](https://dam-flood-water-security.streamlit.app/?stage=mlp) &nbsp;|&nbsp; [Project overview](https://dam-flood-water-security.streamlit.app/?stage=start) &nbsp;|&nbsp; [Next: Wait Versus Act Early](https://dam-flood-water-security.streamlit.app/?stage=compare) ▶

---
# 7. Wait Versus Act Early
### Phase 5 of 6 · Scenario Test

## Part 1 · At the dam
Waiting preserves all water before the storm but may force a large spill at the peak. Acting early creates storage while downstream capacity is still available.

## Part 2 · Engineering challenge
A recommendation is meaningful only if it reduces peak level or emergency spill without unnecessarily sacrificing retained water.

## Part 3 · Where AI comes in
Run the same simulated storm twice and compare peak level, emergency spill, early release, retained storage, and downstream risk.

**Civil Engineering:** Wait Versus Act Early → **AI:** Scenario Simulation → `no pre-release vs constrained AI pre-release`

> 🎬 **See this illustrated and interactive:** [https://dam-flood-water-security.streamlit.app/?stage=compare](https://dam-flood-water-security.streamlit.app/?stage=compare)

## Part 4 · Technical explanation

In [ ]:
def scenario(level,rain,inflow,current_release,pre_release):
 hours=np.arange(7);shape=np.array([.25,.65,1,.85,.55,.30,.12]);path=[level];spill=[]
 for h in range(6):
  gain=(rain/120)*shape[h]*2.25+inflow/850
  early=pre_release/230 if h<3 else 0
  nxt=path[-1]+gain-current_release/300-early
  spill.append(max(0,nxt-100)*100);path.append(min(100,nxt))
 return np.array(path),max(spill+[0])

no_ai,no_spill=scenario(88,120,420,150,0);with_ai,ai_spill=scenario(88,120,420,150,recommended)
comparison=pd.DataFrame({"Metric":["Peak reservoir level","Emergency spill index","Water released before storm","Water retained after storm"],"No pre-release":[f"{no_ai.max():.1f}%",f"{no_spill:.0f}","0 Mm³",f"{no_ai[-1]:.1f}%"],"AI pre-release":[f"{with_ai.max():.1f}%",f"{ai_spill:.0f}",f"{recommended*3*3600/1e6:.2f} Mm³",f"{with_ai[-1]:.1f}%"]})
display(comparison)
plt.figure(figsize=(10,4));plt.plot(no_ai,"o-",label="Without AI");plt.plot(with_ai,"o-",label="With constrained pre-release");plt.axhline(100,color="red",ls="--");plt.xlabel("Hours");plt.ylabel("Reservoir level (%)");plt.grid(alpha=.2);plt.legend();plt.show()

## Part 5 · What you just built

**In the notebook:** Calculate a side-by-side scenario table and storage curves.

**Takeaway:** Judge the policy by both flood protection and water retained after the event.

◀ [Previous: Selecting a Pre-Release](https://dam-flood-water-security.streamlit.app/?stage=decision) &nbsp;|&nbsp; [Project overview](https://dam-flood-water-security.streamlit.app/?stage=start) &nbsp;|&nbsp; [Next: The Dam Safety Audit](https://dam-flood-water-security.streamlit.app/?stage=audit) ▶

---
# 8. The Dam Safety Audit
### Phase 6 of 6 · Engineering Audit

## Part 1 · At the dam
Dam releases are high-consequence decisions governed by operating rules, forecasts, instrumentation, and responsible authorities.

## Part 2 · Engineering challenge
A low average prediction error can hide underprediction during the rare extreme events that matter most.

## Part 3 · Where AI comes in
Audit regression error, high-level underprediction, forecast sensitivity, and rule behaviour separately. Never present this notebook as autonomous control.

**Civil Engineering:** The Dam Safety Audit → **AI:** Model and Policy Evaluation → `MAE, RMSE, high-level misses, sensitivity`

> 🎬 **See this illustrated and interactive:** [https://dam-flood-water-security.streamlit.app/?stage=audit](https://dam-flood-water-security.streamlit.app/?stage=audit)

## Part 4 · Technical explanation

In [ ]:
pred_test=model.predict(X_test,verbose=0).ravel();mae=mean_absolute_error(y_test,pred_test);rmse=mean_squared_error(y_test,pred_test)**.5
extreme=y_test>95;under=((y_test-pred_test)>2)&extreme
print(f"Test MAE: {mae:.2f} percentage points");print(f"Test RMSE: {rmse:.2f} percentage points")
print(f"Extreme events (>95% actual): {extreme.sum()}");print(f"Extreme underpredictions by >2 points: {under.sum()}")
plt.figure(figsize=(6,6));plt.scatter(y_test,pred_test,s=10,alpha=.35);plt.plot([50,103],[50,103],"r--");plt.xlabel("Actual (%)");plt.ylabel("Predicted (%)");plt.grid(alpha=.2);plt.show()
print("Deployment exclusions: no real dam, no calibrated catchment, no forecast ensemble, no rule curve, no emergency-action integration, and no operational authority.")

## Part 5 · What you just built

**In the notebook:** Report test metrics, extreme-event errors, assumptions, and deployment exclusions.

**Takeaway:** A student model can illustrate decision support; it cannot authorize a real dam release.

◀ [Previous: Wait Versus Act Early](https://dam-flood-water-security.streamlit.app/?stage=compare) &nbsp;|&nbsp; [Project overview](https://dam-flood-water-security.streamlit.app/?stage=start)

---
# Final engineering conclusion

The MLP predicts a six-hour reservoir level from six summarized conditions. A separate project-assumption rule proposes a pre-release and caps it when the downstream river is high. The scenario comparison evaluates both flood protection and retained water. This separation keeps the model, operating assumptions, and safety constraints visible.